# Bonus — FP-Growth on Spec Co-occurrence (CSE488 stretch goal, section 4)

Mines frequent itemsets across device specs (CPU tier, RAM bucket, GPU brand, storage type) to find commonly-paired specifications — e.g. "Ryzen 7 + 16GB RAM + SSD" appearing together far more often than chance.

Powers a "commonly paired specifications" insight feature, and is a legitimate Spark MLlib application per the assignment's stretch goal.

## 1. Setup

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.1", "pandas", "pyarrow>=16,<18"])

subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-11-jdk-headless"])
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
subprocess.run(["java", "-version"])

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
from pyspark.ml.fpm import FPGrowth

spark = (
    SparkSession.builder
    .appName("cse488-fpgrowth")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

import pandas as pd
pdf = pd.read_parquet("laptop_chunks_embeddings_with_lineage.parquet")
device_pdf = pdf.drop_duplicates(subset="row_uid").copy()
print(device_pdf.shape)

## 2. Convert raw specs into categorical "items" for FP-Growth

FP-Growth needs discrete items per row (like a shopping basket), not raw numeric/text fields. Bucket each spec into a small number of categories.

In [ ]:
import re

def bucket_ram(ram):
    if pd.isna(ram):
        return None
    if ram < 8:
        return "RAM_<8GB"
    elif ram < 16:
        return "RAM_8-16GB"
    elif ram < 32:
        return "RAM_16-32GB"
    else:
        return "RAM_32GB+"

def bucket_price(price):
    if pd.isna(price):
        return None
    if price < 400:
        return "PRICE_<400"
    elif price < 700:
        return "PRICE_400-700"
    elif price < 1200:
        return "PRICE_700-1200"
    else:
        return "PRICE_1200+"

def bucket_cpu(cpu):
    if not isinstance(cpu, str):
        return None
    cpu_l = cpu.lower()
    if "ryzen 7" in cpu_l or "ryzen 9" in cpu_l:
        return "CPU_Ryzen7/9"
    if "ryzen 5" in cpu_l:
        return "CPU_Ryzen5"
    if "ryzen 3" in cpu_l:
        return "CPU_Ryzen3"
    if "i9" in cpu_l or "i7" in cpu_l:
        return "CPU_i7/i9"
    if "i5" in cpu_l:
        return "CPU_i5"
    if "i3" in cpu_l:
        return "CPU_i3"
    if "core ultra" in cpu_l:
        return "CPU_CoreUltra"
    return "CPU_Other"

def bucket_gpu(gpu):
    if not isinstance(gpu, str):
        return None
    gpu_l = gpu.lower()
    if "rtx" in gpu_l or "nvidia" in gpu_l or "geforce" in gpu_l:
        return "GPU_NVIDIA"
    if "radeon" in gpu_l or "amd" in gpu_l:
        return "GPU_AMD"
    if "intel" in gpu_l:
        return "GPU_IntelIntegrated"
    return "GPU_Other"

def bucket_storage(storage):
    if not isinstance(storage, str):
        return None
    s = storage.lower()
    items = []
    if "1tb" in s or "1 tb" in s:
        items.append("STORAGE_1TB")
    elif "512" in s:
        items.append("STORAGE_512GB")
    elif "256" in s:
        items.append("STORAGE_256GB")
    if "ssd" in s or "pcie" in s or "nvme" in s:
        items.append("STORAGE_SSD")
    if "hdd" in s:
        items.append("STORAGE_HDD")
    return items

def build_basket(row):
    items = []
    for fn, col in [(bucket_ram, "ram_gb"), (bucket_price, "price_usd"), (bucket_cpu, "cpu"), (bucket_gpu, "gpu")]:
        val = fn(row[col])
        if val:
            items.append(val)
    storage_items = bucket_storage(row["storage"])
    if storage_items:
        items.extend(storage_items)
    return items

device_pdf["basket"] = device_pdf.apply(build_basket, axis=1)
device_pdf = device_pdf[device_pdf["basket"].apply(len) >= 2]  # need at least 2 items to find co-occurrence
print(device_pdf[["title", "basket"]].head(5).to_string())
print("devices with usable baskets:", len(device_pdf))

## 3. Run FP-Growth (Spark MLlib)

In [ ]:
basket_df = spark.createDataFrame(
    device_pdf[["row_uid", "basket"]].rename(columns={"basket": "items"})
)

fp = FPGrowth(itemsCol="items", minSupport=0.05, minConfidence=0.3)
model = fp.fit(basket_df)

print("=== Frequent itemsets (specs that commonly appear together) ===")
freq_itemsets = model.freqItemsets.orderBy(F.desc("freq"))
freq_itemsets.show(20, truncate=False)

print("=== Association rules (X -> Y with confidence/lift) ===")
rules = model.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

## 4. Human-readable insights (for the frontend / report)

Turn the raw itemsets into a "commonly paired specifications" feature — e.g. given a CPU tier, what RAM/GPU combos are most associated with it.

In [ ]:
rules_pdf = rules.select("antecedent", "consequent", "confidence", "lift", "support").toPandas()

def format_rule(row):
    ante = ", ".join(row["antecedent"])
    cons = ", ".join(row["consequent"])
    return f"{ante}  ->  {cons}   (confidence={row['confidence']:.2f}, lift={row['lift']:.2f})"

rules_pdf["readable"] = rules_pdf.apply(format_rule, axis=1)
print(rules_pdf[["readable"]].head(15).to_string(index=False))

rules_pdf.to_csv("fpgrowth_association_rules.csv", index=False)

freq_pdf = freq_itemsets.toPandas()
freq_pdf.to_csv("fpgrowth_frequent_itemsets.csv", index=False)

from google.colab import files
files.download("fpgrowth_association_rules.csv")
files.download("fpgrowth_frequent_itemsets.csv")

## 5. (Optional) Wire into the M4 API as a "commonly paired with" feature

Given a retrieved device's CPU/GPU/RAM bucket, look up which other specs commonly co-occur, and surface it as an extra insight in the `/recommend` response — e.g. "laptops with this CPU commonly also have 16-32GB RAM and an SSD."

In [ ]:
def get_paired_specs(item_bucket: str, rules_pdf: pd.DataFrame, top_n: int = 3):
    """Given one spec bucket (e.g. 'CPU_Ryzen7/9'), return the top commonly-paired specs."""
    matches = rules_pdf[rules_pdf["antecedent"].apply(lambda a: item_bucket in a and len(a) == 1)]
    matches = matches.sort_values("lift", ascending=False).head(top_n)
    return [
        {"paired_with": ", ".join(row["consequent"]), "confidence": round(row["confidence"], 2), "lift": round(row["lift"], 2)}
        for _, row in matches.iterrows()
    ]

# example
print(get_paired_specs("CPU_Ryzen7/9", rules_pdf))